# BatchNorm Effect on Distributions

**BatchNorm** re-centers and rescales activations batch-wise.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Layer without BatchNorm

In [ ]:
class PlainNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(32, 64)
    def forward(self, x):
        return self.fc(x)

class BNNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(32, 64)
        self.bn = nn.BatchNorm1d(64)
    def forward(self, x):
        return self.bn(self.fc(x))

X = torch.randn(128, 32) * 3 + 2

## 2. Activation distributions before BN

In [ ]:
plain = PlainNet()
with torch.no_grad():
    h_plain = plain(X)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(h_plain.flatten().numpy(), bins=50, color='coral', edgecolor='black')
ax.set_title('Linear output distribution (shifted, wide)')
plt.tight_layout(); plt.show()

## 3. After BatchNorm

In [ ]:
bn = BNNet()
with torch.no_grad():
    h_bn = bn(X)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(h_bn.flatten().numpy(), bins=50, color='seagreen', edgecolor='black')
ax.set_title('After BatchNorm (~0 mean, ~1 std per feature)')
plt.tight_layout(); plt.show()

## 4. Side-by-side histograms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(h_plain.flatten().numpy(), bins=50, color='coral', alpha=0.8)
axes[0].set_title(f'before: μ={h_plain.mean():.2f}, σ={h_plain.std():.2f}')
axes[1].hist(h_bn.flatten().numpy(), bins=50, color='seagreen', alpha=0.8)
axes[1].set_title(f'after: μ={h_bn.mean():.2f}, σ={h_bn.std():.2f}')
plt.tight_layout(); plt.show()

## 5. Per-feature mean before/after

In [ ]:
feat_mean_before = h_plain.mean(dim=0).numpy()
feat_mean_after = h_bn.mean(dim=0).numpy()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(feat_mean_before, 'o-', label='before BN', color='coral')
ax.plot(feat_mean_after, 's-', label='after BN', color='seagreen')
ax.set_xlabel('feature index'); ax.legend()
ax.set_title('Per-feature batch mean')
plt.tight_layout(); plt.show()